### Análise de Sentimentos em Reviews de Apps Brasileiros ###

Projeto de NLP aplicado a avaliações reais coletadas da Google Play Store,
com foco em texto em português brasileiro.

**Objetivo:** classificar o sentimento de reviews (positivo/neutro/negativo)
a partir do texto e da nota dada pelo usuário.

In [24]:
from google_play_scraper import reviews, Sort
import pandas as pd 
from pathlib import Path


### 1. Coleta de dados ###

Reviews coletados via `google-play-scraper`, app: Nubank (com.nu.production)

In [ ]:
resultado, _ = reviews(
    'com.nu.production',  # app ID do Nubank na Play Store
    lang='pt',
    country='br',
    sort=Sort.NEWEST,
    count=500
)

df = pd.DataFrame(resultado)
print(df.shape)
df.head()


In [26]:
path = Path("../data/raw")
path.mkdir(parents=True, exist_ok=True)

df.to_csv(path / "reviews_nubank.csv", index=False)
print("Salvo com sucesso!")

Salvo com sucesso!


### 2. Exploração inicial ###

Verificando estrutura, tipos de dados e valores faltantes antes de seguir.

In [27]:
print("Colunas:", df.columns.tolist())
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores nulos por coluna:")
print(df.isnull().sum())

Colunas: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

Tipos de dados:
reviewId                           str
userName                           str
userImage                          str
content                            str
score                            int64
thumbsUpCount                    int64
reviewCreatedVersion               str
at                      datetime64[us]
replyContent                       str
repliedAt               datetime64[us]
appVersion                         str
dtype: object

Valores nulos por coluna:
reviewId                  0
userName                  0
userImage                 0
content                   0
score                     0
thumbsUpCount             0
reviewCreatedVersion     55
at                        0
replyContent            444
repliedAt               444
appVersion               55
dtype: int64


In [28]:
print(df_limpo['nota'].value_counts().sort_index())
print("\nMédia das notas:", df_limpo['nota'].mean().round(2))

nota
1     65
2     16
3     31
4     26
5    362
Name: count, dtype: int64

Média das notas: 4.21


### 3. Limpeza de texto ###

Removendo emojis, pontuação, números e padronizando para minúsculo.
A nota (estrelas) será usada como rótulo de sentimento — mais confiável
que tentar inferir sentimento a partir de emojis isoladamente

In [29]:
import re

def limpar_texto(texto):
    texto = texto.lower() #### Tudo minúsculo
    texto = re.sub(r'[^\w\s]', '', texto) #### Remove pontuação
    texto = re.sub(r'\d+', '', texto) #### Remove números
    texto = re.sub(r'\s+', ' ', texto).strip() #### Remove espaços extras
    return texto

df_limpo['texto_limpo'] = df_limpo['texto'].apply(limpar_texto)
df_limpo[['texto', 'texto_limpo']].head(5)

,texto,texto_limpo
0,Quero ver abrir meu aplicativo,quero ver abrir meu aplicativo
1,Boa demqis,boa demqis
2,maravilhosa de mais ❤️,maravilhosa de mais
3,Excelente 👌,excelente
4,Não aparece mais o dinheiro que eu envio ao po...,não aparece mais o dinheiro que eu envio ao po...


### 4. Rotulagem de sentimento ###

Classificação baseada na nota:
-> 1-2 estrelas → negativo
-> 3 estrelas → neutro
-> 4-5 estrelas → positivo

In [30]:
def classificar_sentimento(nota):
    if nota <= 2:
        return "negativo"
    elif nota == 3:
        return "neutro"
    else:
        return "positivo"
    
    df_limpo['sentimento'] = df_limpo['nota'].apply(classificar_sentimento)
    df_limpo['sentimento'].value_counts()

In [31]:
path_processed = Path("../data/processed")
path_processed.mkdir(parents=True, exist_ok=True)
df_limpo.to_csv(path_processed / "reviews_nubank_limpo.csv", index=False)
print("Dataset processado salvo!")

Dataset processado salvo!


In [32]:
df_limpo = df[['userName', 'score', 'content', 'at']].copy()
df_limpo.columns = ['usuario', 'nota', 'texto', 'data']
df_limpo.head(10)

,usuario,nota,texto,data
0,Leandro Solon,5,Quero ver abrir meu aplicativo,2026-06-20 01:35:38
1,Wagner Monteiro,5,Boa demqis,2026-06-20 01:33:03
2,Lucia Andrade,5,maravilhosa de mais ❤️,2026-06-20 01:32:54
3,Ageu Ferreira,5,Excelente 👌,2026-06-20 00:00:04
4,Duda Moreira,1,Não aparece mais o dinheiro que eu envio ao po...,2026-06-19 23:21:55
5,Ana Elizabete Santos Borges,5,muito bom,2026-06-19 23:14:36
6,Márcio Daniel,5,o banco do futuro!,2026-06-19 22:40:54
7,Eronilda Lira,3,eu queria fazer um empréstimo. mas os juros sã...,2026-06-19 22:37:03
8,kaua Fernando,5,o melhor banco que tem,2026-06-19 22:34:55
9,Marcelo Galdino,5,ótimo cartão,2026-06-19 22:05:35


In [35]:
for texto in df_limpo['texto'].head(5):
    print(texto)
    print("-" * 50)

Quero ver abrir meu aplicativo
--------------------------------------------------
Boa demqis
--------------------------------------------------
maravilhosa de mais ❤️
--------------------------------------------------
Excelente 👌
--------------------------------------------------
Não aparece mais o dinheiro que eu envio ao povo! Acaba ficando ruim, pois eu me perco sem saber com o que eu gastei.
--------------------------------------------------


### 5. Visualização dos dados ###

Gráficos para entender a distribuição de notas e sentimentos.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")

In [ ]:
plt.figure(figsize=(8, 5))
ordem = ['negativo', 'neutro', 'positivo']
cores = ['#E74C3C', '#F1C40F', '#2ECC71']

sns.countplot(data=df_limpo, x='sentimento', order=ordem, palette=cores)
plt.title('Distribuição de Sentimento - Reviews do Nubank', fontsize=14)
plt.xlabel('Sentimento')
plt.ylabel('Quantidade de Reviews')
plt.tight_layout()
plt.savefig('../data/processed/grafico_sentimentos.png', dpi=150)
plt.show()

SyntaxError: invalid syntax (980335849.py, line 10)

In [ ]:
from collections import Counter

def palavras_mais_comuns(df, sentimento, n=10):
    textos = df[df['sentimento'] == sentimento]['texto_limpo']
    palavras = ' '.join(textos).split()
    contagem = Counter(palavras)
    return contagem.most_common(n)

print("Top palavras - NEGATIVO:")
print(palavras_mais_comuns(df_limpo, 'negativo'))
print("\nTop palavras - POSITIVO:")
print(palavras_mais_comuns(df_limpo, 'positivo'))

